# GPT Training and GPU Benchmark

This notebook trains a **decoder-only GPT transformer from scratch** and records
enough instrumentation to function as a repeatable GPU benchmark.

## Machine-learning workload

- Multi-layer causal transformer
- Scaled dot-product attention
- BF16 or FP16 mixed-precision training
- Gradient accumulation
- AdamW optimization and learning-rate scheduling
- Training and validation loss
- Checkpointing and automatic resume
- Autoregressive text generation

## Benchmarking and statistics

The notebook now records:

- End-to-end optimizer-step time
- GPU-only optimizer-step time using CUDA events
- Mean, median, minimum, maximum, standard deviation, P90 and P95 step times
- Wall-clock and GPU tokens per second
- Approximate training TFLOP/s
- Forward-pass latency
- Forward-and-backward latency
- Validation duration and validation throughput
- Generation latency and generated tokens per second
- Allocated, reserved and peak VRAM
- NVIDIA GPU utilization, temperature, power and memory utilization
- Checkpoint write duration and checkpoint file size
- Total session time and total tokens processed
- CSV exports and charts for later comparisons

The default dataset is **Tiny Shakespeare**. Replace it with any local `.txt`
file by changing `LOCAL_TEXT_FILE`.

## Long-running behavior

The default is `MAX_STEPS = 10_000`, but there is no 60-second timeout in the
training logic. Set `RUN_FOREVER = True` to continue until you interrupt the
training cell. Interruption triggers a checkpoint before the cell exits.

Keep this running in another PowerShell window:

```powershell
nvidia-smi -l 1
```

## 1. Imports and GPU diagnostics

In [ ]:
import os
import sys
import time
import math
import json
import random
import platform
import subprocess
import statistics
import urllib.request
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Confirm that this notebook is using "
        "a CUDA-enabled PyTorch installation."
    )

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
total_vram_gb = gpu_props.total_memory / 1024**3

print("GPU:", gpu_name)
print("Compute capability:", torch.cuda.get_device_capability(0))
print("VRAM:", round(total_vram_gb, 2), "GB")
print("Multiprocessors:", gpu_props.multi_processor_count)
print("Platform:", platform.platform())

torch.manual_seed(1337)
torch.cuda.manual_seed_all(1337)
random.seed(1337)

# Allows TensorFloat-32 acceleration for applicable FP32 operations.
torch.set_float32_matmul_precision("high")

## 2. Training configuration

In [ ]:
# -------------------- Main controls --------------------

PROFILE = "auto"          # auto, small, medium, large, xl
MAX_STEPS = 10_000        # Increase for a longer benchmark.
RUN_FOREVER = False       # True = train until manually interrupted.

EVAL_INTERVAL = 250
EVAL_BATCHES = 30
LOG_INTERVAL = 25
CHECKPOINT_INTERVAL = 500
GENERATE_INTERVAL = 1_000
GENERATION_SAMPLE_TOKENS = 300

LEARNING_RATE = 3e-4
MIN_LEARNING_RATE = 3e-5
WARMUP_STEPS = 200
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
DROPOUT = 0.1

# Model-specific preflight benchmark.
BENCHMARK_WARMUP_ITERATIONS = 5
BENCHMARK_TIMED_ITERATIONS = 25
INFERENCE_BENCHMARK_TOKENS = 200

# Leave blank to download Tiny Shakespeare.
# Example: LOCAL_TEXT_FILE = r"C:\Users\YOUR_USERNAME\AI\my_training_text.txt"
LOCAL_TEXT_FILE = ""

DATA_DIR = Path("./gpt_data")
OUTPUT_DIR = Path("./gpt_output")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "gpt_checkpoint.pt"
EVAL_METRICS_PATH = OUTPUT_DIR / "evaluation_metrics.csv"
STEP_METRICS_PATH = OUTPUT_DIR / "step_metrics.csv"
GENERATION_METRICS_PATH = OUTPUT_DIR / "generation_metrics.csv"
SYSTEM_METRICS_PATH = OUTPUT_DIR / "system_metrics.csv"
CHECKPOINT_METRICS_PATH = OUTPUT_DIR / "checkpoint_metrics.csv"
MICROBENCHMARK_PATH = OUTPUT_DIR / "model_microbenchmarks.csv"
SUMMARY_PATH = OUTPUT_DIR / "benchmark_summary.csv"

# torch.compile can improve performance after a potentially long first compile.
# It can be less predictable on some Windows installations, so it is off by default.
USE_TORCH_COMPILE = False

# -------------------- VRAM-aware profiles --------------------

profiles = {
    "small": {
        "n_layer": 6,
        "n_head": 6,
        "n_embd": 384,
        "block_size": 256,
        "micro_batch_size": 32,
        "gradient_accumulation_steps": 4,
    },
    "medium": {
        "n_layer": 10,
        "n_head": 10,
        "n_embd": 640,
        "block_size": 384,
        "micro_batch_size": 20,
        "gradient_accumulation_steps": 4,
    },
    "large": {
        "n_layer": 12,
        "n_head": 12,
        "n_embd": 768,
        "block_size": 512,
        "micro_batch_size": 24,
        "gradient_accumulation_steps": 4,
    },
    "xl": {
        "n_layer": 16,
        "n_head": 16,
        "n_embd": 1024,
        "block_size": 512,
        "micro_batch_size": 24,
        "gradient_accumulation_steps": 4,
    },
}

if PROFILE == "auto":
    if total_vram_gb < 10:
        selected_profile = "small"
    elif total_vram_gb < 18:
        selected_profile = "medium"
    elif total_vram_gb < 32:
        selected_profile = "large"
    else:
        selected_profile = "xl"
else:
    selected_profile = PROFILE

config = profiles[selected_profile].copy()

effective_sequences_per_step = (
    config["micro_batch_size"]
    * config["gradient_accumulation_steps"]
)
effective_tokens_per_step = (
    effective_sequences_per_step
    * config["block_size"]
)

print("Selected profile:", selected_profile)
print(json.dumps(config, indent=2))
print("Effective sequences per optimizer step:", effective_sequences_per_step)
print("Effective tokens per optimizer step:", f"{effective_tokens_per_step:,}")


### Making it heavier or lighter

The main GPU-load control is `micro_batch_size`.

- Increase it gradually to consume more VRAM and improve utilization.
- Reduce it if you receive a CUDA out-of-memory error.
- `gradient_accumulation_steps` increases the effective batch size without
  requiring all of it to fit in VRAM simultaneously.
- The `large` and `xl` profiles are intentionally substantial.


## 3. Load and encode the training text

In [ ]:

TINY_SHAKESPEARE_URL = (
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/"
    "tinyshakespeare/input.txt"
)
downloaded_path = DATA_DIR / "tinyshakespeare.txt"

if LOCAL_TEXT_FILE:
    text_path = Path(LOCAL_TEXT_FILE)
    if not text_path.exists():
        raise FileNotFoundError(f"Local text file not found: {text_path}")
else:
    text_path = downloaded_path
    if not text_path.exists():
        print("Downloading Tiny Shakespeare...")
        try:
            urllib.request.urlretrieve(TINY_SHAKESPEARE_URL, text_path)
        except Exception as exc:
            raise RuntimeError(
                "The dataset download failed, possibly because of the network. "
                "Download a plain-text dataset manually "
                "and set LOCAL_TEXT_FILE to its path."
            ) from exc

text = text_path.read_text(encoding="utf-8")
print("Dataset:", text_path.resolve())
print("Characters:", f"{len(text):,}")
print("Preview:\n")
print(text[:500])


In [ ]:

characters = sorted(set(text))
vocab_size = len(characters)

stoi = {character: index for index, character in enumerate(characters)}
itos = {index: character for character, index in stoi.items()}

def encode(value):
    return [stoi[character] for character in value]

def decode(indices):
    return "".join(itos[int(index)] for index in indices)

data = torch.tensor(encode(text), dtype=torch.long)
split_index = int(0.9 * len(data))
train_data = data[:split_index]
val_data = data[split_index:]

print("Vocabulary size:", vocab_size)
print("Training tokens:", f"{len(train_data):,}")
print("Validation tokens:", f"{len(val_data):,}")


## 4. Batch generation

In [ ]:

block_size = config["block_size"]
micro_batch_size = config["micro_batch_size"]

def get_batch(split):
    source = train_data if split == "train" else val_data

    starts = torch.randint(
        low=0,
        high=len(source) - block_size - 1,
        size=(micro_batch_size,),
    )

    x = torch.stack([
        source[start : start + block_size]
        for start in starts
    ])
    y = torch.stack([
        source[start + 1 : start + block_size + 1]
        for start in starts
    ])

    # Non-blocking transfers become useful if tensors are pinned.
    x = x.pin_memory().to(device, non_blocking=True)
    y = y.pin_memory().to(device, non_blocking=True)
    return x, y

x_test, y_test = get_batch("train")
print("Input shape:", tuple(x_test.shape))
print("Target shape:", tuple(y_test.shape))
print("Input device:", x_test.device)

del x_test, y_test
torch.cuda.empty_cache()


## 5. GPT model

In [ ]:

@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0
    bias: bool = False


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        if cfg.n_embd % cfg.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")

        self.n_head = cfg.n_head
        self.head_dim = cfg.n_embd // cfg.n_head
        self.dropout = cfg.dropout

        self.qkv = nn.Linear(
            cfg.n_embd,
            3 * cfg.n_embd,
            bias=cfg.bias,
        )
        self.projection = nn.Linear(
            cfg.n_embd,
            cfg.n_embd,
            bias=cfg.bias,
        )
        self.residual_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        batch, sequence, channels = x.shape

        q, k, v = self.qkv(x).split(channels, dim=2)

        q = q.view(
            batch,
            sequence,
            self.n_head,
            self.head_dim,
        ).transpose(1, 2)

        k = k.view(
            batch,
            sequence,
            self.n_head,
            self.head_dim,
        ).transpose(1, 2)

        v = v.view(
            batch,
            sequence,
            self.n_head,
            self.head_dim,
        ).transpose(1, 2)

        # PyTorch dispatches to an optimized attention implementation when possible.
        attention = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )

        attention = attention.transpose(1, 2).contiguous().view(
            batch,
            sequence,
            channels,
        )

        return self.residual_dropout(self.projection(attention))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.up_projection = nn.Linear(
            cfg.n_embd,
            4 * cfg.n_embd,
            bias=cfg.bias,
        )
        self.activation = nn.GELU(approximate="tanh")
        self.down_projection = nn.Linear(
            4 * cfg.n_embd,
            cfg.n_embd,
            bias=cfg.bias,
        )
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        x = self.up_projection(x)
        x = self.activation(x)
        x = self.down_projection(x)
        return self.dropout(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm_1 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.attention = CausalSelfAttention(cfg)
        self.norm_2 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attention(self.norm_1(x))
        x = x + self.mlp(self.norm_2(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        self.token_embedding = nn.Embedding(
            cfg.vocab_size,
            cfg.n_embd,
        )
        self.position_embedding = nn.Embedding(
            cfg.block_size,
            cfg.n_embd,
        )
        self.embedding_dropout = nn.Dropout(cfg.dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(cfg)
            for _ in range(cfg.n_layer)
        ])

        self.final_norm = nn.LayerNorm(
            cfg.n_embd,
            bias=cfg.bias,
        )
        self.language_model_head = nn.Linear(
            cfg.n_embd,
            cfg.vocab_size,
            bias=False,
        )

        # Weight tying reduces parameters and is standard in many language models.
        self.language_model_head.weight = self.token_embedding.weight

        self.apply(self._initialize_weights)

        # Scale residual projections according to depth.
        for name, parameter in self.named_parameters():
            if name.endswith("projection.weight") or name.endswith("down_projection.weight"):
                nn.init.normal_(
                    parameter,
                    mean=0.0,
                    std=0.02 / math.sqrt(2 * cfg.n_layer),
                )

    def _initialize_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, token_indices, targets=None):
        batch, sequence = token_indices.shape

        if sequence > self.cfg.block_size:
            raise ValueError(
                f"Sequence length {sequence} exceeds block size "
                f"{self.cfg.block_size}"
            )

        positions = torch.arange(
            0,
            sequence,
            device=token_indices.device,
        )

        x = self.token_embedding(token_indices)
        x = x + self.position_embedding(positions)
        x = self.embedding_dropout(x)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)
        logits = self.language_model_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
            )

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        token_indices,
        max_new_tokens,
        temperature=0.8,
        top_k=40,
    ):
        self.eval()

        for _ in range(max_new_tokens):
            context = token_indices[:, -self.cfg.block_size :]
            logits, _ = self(context)
            logits = logits[:, -1, :] / max(temperature, 1e-5)

            if top_k is not None:
                values, _ = torch.topk(
                    logits,
                    min(top_k, logits.size(-1)),
                )
                cutoff = values[:, [-1]]
                logits = logits.masked_fill(logits < cutoff, float("-inf"))

            probabilities = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(
                probabilities,
                num_samples=1,
            )
            token_indices = torch.cat(
                (token_indices, next_token),
                dim=1,
            )

        return token_indices


## 6. Build the model and optimizer

In [ ]:

model_config = GPTConfig(
    vocab_size=vocab_size,
    block_size=config["block_size"],
    n_layer=config["n_layer"],
    n_head=config["n_head"],
    n_embd=config["n_embd"],
    dropout=DROPOUT,
    bias=False,
)

model = GPT(model_config).to(device)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
trainable_count = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Model configuration:")
print(json.dumps(asdict(model_config), indent=2))
print("Parameters:", f"{parameter_count:,}")
print("Trainable parameters:", f"{trainable_count:,}")

if USE_TORCH_COMPILE:
    print("Compiling the model. The first iteration may take a while...")
    model = torch.compile(model)

# Prefer fused AdamW when supported by the installed PyTorch/CUDA combination.
try:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        betas=(0.9, 0.95),
        eps=1e-8,
        weight_decay=WEIGHT_DECAY,
        fused=True,
    )
    optimizer_type = "fused AdamW"
except Exception:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        betas=(0.9, 0.95),
        eps=1e-8,
        weight_decay=WEIGHT_DECAY,
    )
    optimizer_type = "standard AdamW"

print("Optimizer:", optimizer_type)

amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)
use_fp16_scaler = amp_dtype == torch.float16
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_fp16_scaler,
)

print("Mixed precision dtype:", amp_dtype)
print("Gradient scaler enabled:", scaler.is_enabled())

torch.cuda.reset_peak_memory_stats()


## 7. Evaluation and scheduling helpers

In [ ]:
def percentile(values, q):
    if not values:
        return float("nan")
    series = pd.Series(values, dtype="float64")
    return float(series.quantile(q))


def distribution_stats(values, prefix):
    values = [float(value) for value in values]
    if not values:
        return {}

    return {
        f"{prefix}_count": len(values),
        f"{prefix}_mean": statistics.fmean(values),
        f"{prefix}_median": statistics.median(values),
        f"{prefix}_min": min(values),
        f"{prefix}_max": max(values),
        f"{prefix}_stdev": (
            statistics.stdev(values) if len(values) > 1 else 0.0
        ),
        f"{prefix}_p90": percentile(values, 0.90),
        f"{prefix}_p95": percentile(values, 0.95),
    }


def query_nvidia_smi():
    query = (
        "utilization.gpu,temperature.gpu,power.draw,power.limit,"
        "memory.used,memory.total,clocks.sm,clocks.mem"
    )
    command = [
        "nvidia-smi",
        f"--query-gpu={query}",
        "--format=csv,noheader,nounits",
    ]

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=True,
            timeout=10,
        )
        first_gpu = result.stdout.strip().splitlines()[0]
        values = [value.strip() for value in first_gpu.split(",")]

        keys = [
            "gpu_utilization_percent",
            "temperature_c",
            "power_draw_w",
            "power_limit_w",
            "memory_used_mb",
            "memory_total_mb",
            "sm_clock_mhz",
            "memory_clock_mhz",
        ]

        parsed = {}
        for key, value in zip(keys, values):
            try:
                parsed[key] = float(value)
            except ValueError:
                parsed[key] = float("nan")

        return parsed

    except Exception as exc:
        return {
            "nvidia_smi_error": str(exc),
        }


@torch.no_grad()
def estimate_loss():
    model.eval()
    output = {}
    total_eval_tokens = 0

    torch.cuda.synchronize()
    wall_start = time.perf_counter()
    gpu_start = torch.cuda.Event(enable_timing=True)
    gpu_end = torch.cuda.Event(enable_timing=True)
    gpu_start.record()

    for split in ("train", "val"):
        losses = torch.zeros(EVAL_BATCHES)

        for index in range(EVAL_BATCHES):
            x, y = get_batch(split)
            total_eval_tokens += x.numel()

            with torch.autocast(
                device_type="cuda",
                dtype=amp_dtype,
            ):
                _, loss = model(x, y)

            losses[index] = loss.detach().float().cpu()

        output[split] = losses.mean().item()

    gpu_end.record()
    torch.cuda.synchronize()

    wall_seconds = time.perf_counter() - wall_start
    gpu_seconds = gpu_start.elapsed_time(gpu_end) / 1000

    model.train()

    output.update({
        "wall_seconds": wall_seconds,
        "gpu_seconds": gpu_seconds,
        "tokens_evaluated": total_eval_tokens,
        "wall_tokens_per_second": (
            total_eval_tokens / max(wall_seconds, 1e-9)
        ),
        "gpu_tokens_per_second": (
            total_eval_tokens / max(gpu_seconds, 1e-9)
        ),
    })

    return output


def learning_rate_for_step(step):
    if step < WARMUP_STEPS:
        return LEARNING_RATE * (step + 1) / max(1, WARMUP_STEPS)

    if not RUN_FOREVER and step >= MAX_STEPS:
        return MIN_LEARNING_RATE

    decay_end = max(MAX_STEPS, WARMUP_STEPS + 1)
    decay_ratio = (step - WARMUP_STEPS) / (decay_end - WARMUP_STEPS)
    decay_ratio = min(max(decay_ratio, 0.0), 1.0)

    coefficient = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return MIN_LEARNING_RATE + coefficient * (
        LEARNING_RATE - MIN_LEARNING_RATE
    )


def histories_to_csv(histories):
    path_map = {
        "steps": STEP_METRICS_PATH,
        "evaluations": EVAL_METRICS_PATH,
        "generations": GENERATION_METRICS_PATH,
        "system": SYSTEM_METRICS_PATH,
        "checkpoints": CHECKPOINT_METRICS_PATH,
        "microbenchmarks": MICROBENCHMARK_PATH,
    }

    for key, path in path_map.items():
        records = histories.get(key, [])
        if records:
            pd.DataFrame(records).to_csv(path, index=False)


def save_checkpoint(step, histories, reason="scheduled"):
    raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model

    checkpoint_started = time.perf_counter()

    checkpoint = {
        "step": step,
        "reason": reason,
        "saved_at": datetime.now().isoformat(timespec="seconds"),
        "model_config": asdict(model_config),
        "model_state_dict": raw_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "histories": histories,
        "characters": characters,
        "selected_profile": selected_profile,
        "training_config": {
            "learning_rate": LEARNING_RATE,
            "minimum_learning_rate": MIN_LEARNING_RATE,
            "warmup_steps": WARMUP_STEPS,
            "weight_decay": WEIGHT_DECAY,
            "gradient_accumulation_steps": config[
                "gradient_accumulation_steps"
            ],
        },
    }

    torch.save(checkpoint, CHECKPOINT_PATH)
    checkpoint_seconds = time.perf_counter() - checkpoint_started
    checkpoint_size_mb = CHECKPOINT_PATH.stat().st_size / 1024**2

    checkpoint_record = {
        "step": step,
        "reason": reason,
        "checkpoint_seconds": checkpoint_seconds,
        "checkpoint_size_mb": checkpoint_size_mb,
        "saved_at": checkpoint["saved_at"],
    }
    histories.setdefault("checkpoints", []).append(checkpoint_record)

    histories_to_csv(histories)

    print(
        f"Checkpoint saved at step {step:,} ({reason}) | "
        f"{checkpoint_seconds:.2f}s | {checkpoint_size_mb:.1f} MB"
    )

    return checkpoint_record


def empty_histories():
    return {
        "steps": [],
        "evaluations": [],
        "generations": [],
        "system": [],
        "checkpoints": [],
        "microbenchmarks": [],
    }


def load_checkpoint_if_available():
    if not CHECKPOINT_PATH.exists():
        return 0, empty_histories()

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=device,
        weights_only=False,
    )

    saved_config = checkpoint.get("model_config")
    current_config = asdict(model_config)

    if saved_config != current_config:
        print("A checkpoint exists, but its model configuration differs.")
        print("Starting a new training run instead of loading it.")
        return 0, empty_histories()

    raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
    raw_model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if checkpoint.get("scaler_state_dict"):
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    step = int(checkpoint["step"]) + 1
    histories = checkpoint.get("histories")

    # Backward compatibility with the first notebook version.
    if histories is None:
        histories = empty_histories()
        legacy_metrics = checkpoint.get("metrics", [])
        histories["evaluations"] = legacy_metrics

    for key in empty_histories():
        histories.setdefault(key, [])

    print(f"Resumed from checkpoint at step {step:,}.")
    return step, histories

## 8. Short readiness test

In [ ]:
model.train()
optimizer.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()

x, y = get_batch("train")

torch.cuda.synchronize()
wall_start = time.perf_counter()
gpu_start = torch.cuda.Event(enable_timing=True)
gpu_end = torch.cuda.Event(enable_timing=True)
gpu_start.record()

with torch.autocast(
    device_type="cuda",
    dtype=amp_dtype,
):
    _, test_loss = model(x, y)

scaler.scale(test_loss).backward()

gpu_end.record()
torch.cuda.synchronize()
wall_seconds = time.perf_counter() - wall_start
gpu_seconds = gpu_start.elapsed_time(gpu_end) / 1000

optimizer.zero_grad(set_to_none=True)

print("Readiness test loss:", round(float(test_loss.detach()), 4))
print("Forward + backward wall time:", round(wall_seconds * 1000, 2), "ms")
print("Forward + backward GPU time:", round(gpu_seconds * 1000, 2), "ms")
print(
    "Allocated VRAM:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)
print(
    "Peak allocated VRAM:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB",
)
print("The model is ready to train.")

del x, y, test_loss
torch.cuda.empty_cache()

## 9. Model-specific preflight benchmark

This benchmark uses the actual GPT rather than an unrelated synthetic matrix.

It measures:

- Forward-pass latency
- Forward-and-backward latency
- Tokens per second
- Approximate training TFLOP/s
- Generation speed
- Peak VRAM

The first few iterations are discarded as warm-up. The reported distributions
use the subsequent timed iterations.

In [ ]:
microbenchmark_results = []

def record_latency_benchmark(
    benchmark_name,
    wall_times_ms,
    gpu_times_ms,
    tokens_per_iteration,
    approximate_flops_per_iteration=None,
):
    wall_stats = distribution_stats(wall_times_ms, "wall_ms")
    gpu_stats = distribution_stats(gpu_times_ms, "gpu_ms")

    median_wall_seconds = wall_stats["wall_ms_median"] / 1000
    median_gpu_seconds = gpu_stats["gpu_ms_median"] / 1000

    record = {
        "benchmark": benchmark_name,
        "iterations": len(wall_times_ms),
        "tokens_per_iteration": tokens_per_iteration,
        "median_wall_tokens_per_second": (
            tokens_per_iteration / max(median_wall_seconds, 1e-9)
        ),
        "median_gpu_tokens_per_second": (
            tokens_per_iteration / max(median_gpu_seconds, 1e-9)
        ),
        "peak_vram_gb": (
            torch.cuda.max_memory_allocated() / 1024**3
        ),
        **wall_stats,
        **gpu_stats,
    }

    if approximate_flops_per_iteration is not None:
        record["approximate_median_gpu_tflops"] = (
            approximate_flops_per_iteration
            / max(median_gpu_seconds, 1e-9)
            / 1e12
        )

    microbenchmark_results.append(record)
    return record


# ---------- Forward-only benchmark ----------

model.eval()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

for _ in range(BENCHMARK_WARMUP_ITERATIONS):
    x, _ = get_batch("train")
    with torch.no_grad(), torch.autocast(
        device_type="cuda",
        dtype=amp_dtype,
    ):
        model(x)
torch.cuda.synchronize()

forward_wall_ms = []
forward_gpu_ms = []

for _ in range(BENCHMARK_TIMED_ITERATIONS):
    x, _ = get_batch("train")

    torch.cuda.synchronize()
    wall_start = time.perf_counter()
    gpu_start = torch.cuda.Event(enable_timing=True)
    gpu_end = torch.cuda.Event(enable_timing=True)

    gpu_start.record()
    with torch.no_grad(), torch.autocast(
        device_type="cuda",
        dtype=amp_dtype,
    ):
        model(x)
    gpu_end.record()

    torch.cuda.synchronize()
    forward_wall_ms.append(
        (time.perf_counter() - wall_start) * 1000
    )
    forward_gpu_ms.append(gpu_start.elapsed_time(gpu_end))

forward_tokens = micro_batch_size * block_size

forward_record = record_latency_benchmark(
    benchmark_name="GPT forward pass",
    wall_times_ms=forward_wall_ms,
    gpu_times_ms=forward_gpu_ms,
    tokens_per_iteration=forward_tokens,
)

# ---------- Forward + backward benchmark ----------

model.train()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

for _ in range(BENCHMARK_WARMUP_ITERATIONS):
    optimizer.zero_grad(set_to_none=True)
    x, y = get_batch("train")
    with torch.autocast(
        device_type="cuda",
        dtype=amp_dtype,
    ):
        _, loss = model(x, y)
    scaler.scale(loss).backward()
torch.cuda.synchronize()
optimizer.zero_grad(set_to_none=True)

train_wall_ms = []
train_gpu_ms = []

for _ in range(BENCHMARK_TIMED_ITERATIONS):
    optimizer.zero_grad(set_to_none=True)
    x, y = get_batch("train")

    torch.cuda.synchronize()
    wall_start = time.perf_counter()
    gpu_start = torch.cuda.Event(enable_timing=True)
    gpu_end = torch.cuda.Event(enable_timing=True)

    gpu_start.record()
    with torch.autocast(
        device_type="cuda",
        dtype=amp_dtype,
    ):
        _, loss = model(x, y)
    scaler.scale(loss).backward()
    gpu_end.record()

    torch.cuda.synchronize()
    train_wall_ms.append(
        (time.perf_counter() - wall_start) * 1000
    )
    train_gpu_ms.append(gpu_start.elapsed_time(gpu_end))

optimizer.zero_grad(set_to_none=True)

# A commonly used rough transformer-training estimate is about
# 6 * parameter_count * tokens floating-point operations.
approximate_training_flops = (
    6 * parameter_count * forward_tokens
)

train_record = record_latency_benchmark(
    benchmark_name="GPT forward + backward",
    wall_times_ms=train_wall_ms,
    gpu_times_ms=train_gpu_ms,
    tokens_per_iteration=forward_tokens,
    approximate_flops_per_iteration=approximate_training_flops,
)

# ---------- Autoregressive inference benchmark ----------

model.eval()
seed = torch.tensor(
    [[stoi.get("\n", 0)]],
    dtype=torch.long,
    device=device,
)

torch.cuda.synchronize()
generation_wall_start = time.perf_counter()
generation_gpu_start = torch.cuda.Event(enable_timing=True)
generation_gpu_end = torch.cuda.Event(enable_timing=True)
generation_gpu_start.record()

generated = model.generate(
    seed,
    max_new_tokens=INFERENCE_BENCHMARK_TOKENS,
    temperature=0.8,
    top_k=40,
)

generation_gpu_end.record()
torch.cuda.synchronize()

generation_wall_seconds = (
    time.perf_counter() - generation_wall_start
)
generation_gpu_seconds = (
    generation_gpu_start.elapsed_time(generation_gpu_end) / 1000
)

generation_record = {
    "benchmark": "Autoregressive generation",
    "iterations": 1,
    "generated_tokens": INFERENCE_BENCHMARK_TOKENS,
    "wall_seconds": generation_wall_seconds,
    "gpu_seconds": generation_gpu_seconds,
    "wall_tokens_per_second": (
        INFERENCE_BENCHMARK_TOKENS
        / max(generation_wall_seconds, 1e-9)
    ),
    "gpu_tokens_per_second": (
        INFERENCE_BENCHMARK_TOKENS
        / max(generation_gpu_seconds, 1e-9)
    ),
    "peak_vram_gb": (
        torch.cuda.max_memory_allocated() / 1024**3
    ),
}
microbenchmark_results.append(generation_record)

microbenchmark_df = pd.DataFrame(microbenchmark_results)
microbenchmark_df.to_csv(MICROBENCHMARK_PATH, index=False)
display(microbenchmark_df)

print("\nPreflight system snapshot:")
print(query_nvidia_smi())

model.train()
del seed, generated
torch.cuda.empty_cache()

## 10. Main training loop

This is the long-running machine-learning benchmark.

Every optimizer step records wall-clock time, GPU-event time, throughput,
approximate TFLOP/s, loss and VRAM. Periodic system samples add utilization,
temperature, power and clock data from `nvidia-smi`.

The cell resumes from a compatible checkpoint and saves benchmark history when
training is interrupted.

In [ ]:
start_step, histories = load_checkpoint_if_available()

# Preserve current preflight results in the saved benchmark history.
if microbenchmark_results:
    histories["microbenchmarks"] = microbenchmark_results

gradient_accumulation_steps = config[
    "gradient_accumulation_steps"
]

model.train()
training_start_time = time.perf_counter()
session_tokens_processed = 0
step = start_step

print(
    f"Starting training at step {step:,}. "
    f"Target: {'manual interruption' if RUN_FOREVER else f'{MAX_STEPS:,} steps'}"
)
print("Effective tokens per optimizer step:", f"{effective_tokens_per_step:,}")

try:
    while RUN_FOREVER or step < MAX_STEPS:
        optimizer.zero_grad(set_to_none=True)

        learning_rate = learning_rate_for_step(step)
        for group in optimizer.param_groups:
            group["lr"] = learning_rate

        accumulated_loss = 0.0

        torch.cuda.synchronize()
        step_wall_start = time.perf_counter()
        step_gpu_start = torch.cuda.Event(enable_timing=True)
        step_gpu_end = torch.cuda.Event(enable_timing=True)
        step_gpu_start.record()

        for micro_step in range(gradient_accumulation_steps):
            x, y = get_batch("train")

            with torch.autocast(
                device_type="cuda",
                dtype=amp_dtype,
            ):
                _, loss = model(x, y)
                scaled_loss = loss / gradient_accumulation_steps

            accumulated_loss += float(loss.detach())
            scaler.scale(scaled_loss).backward()

        if GRAD_CLIP is not None:
            scaler.unscale_(optimizer)
            gradient_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP,
            )
            gradient_norm_value = float(gradient_norm.detach().cpu())
        else:
            gradient_norm_value = float("nan")

        scaler.step(optimizer)
        scaler.update()

        step_gpu_end.record()
        torch.cuda.synchronize()

        step_wall_seconds = (
            time.perf_counter() - step_wall_start
        )
        step_gpu_seconds = (
            step_gpu_start.elapsed_time(step_gpu_end) / 1000
        )

        mean_training_loss = (
            accumulated_loss / gradient_accumulation_steps
        )

        session_tokens_processed += effective_tokens_per_step

        wall_tokens_per_second = (
            effective_tokens_per_step
            / max(step_wall_seconds, 1e-9)
        )
        gpu_tokens_per_second = (
            effective_tokens_per_step
            / max(step_gpu_seconds, 1e-9)
        )

        # Approximation for transformer training, useful for relative comparisons.
        approximate_step_flops = (
            6 * parameter_count * effective_tokens_per_step
        )
        approximate_wall_tflops = (
            approximate_step_flops
            / max(step_wall_seconds, 1e-9)
            / 1e12
        )
        approximate_gpu_tflops = (
            approximate_step_flops
            / max(step_gpu_seconds, 1e-9)
            / 1e12
        )

        step_record = {
            "step": step,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "training_loss": mean_training_loss,
            "learning_rate": learning_rate,
            "gradient_norm": gradient_norm_value,
            "wall_step_ms": step_wall_seconds * 1000,
            "gpu_step_ms": step_gpu_seconds * 1000,
            "wall_tokens_per_second": wall_tokens_per_second,
            "gpu_tokens_per_second": gpu_tokens_per_second,
            "approximate_wall_tflops": approximate_wall_tflops,
            "approximate_gpu_tflops": approximate_gpu_tflops,
            "effective_tokens": effective_tokens_per_step,
            "allocated_vram_gb": (
                torch.cuda.memory_allocated() / 1024**3
            ),
            "reserved_vram_gb": (
                torch.cuda.memory_reserved() / 1024**3
            ),
            "peak_vram_gb": (
                torch.cuda.max_memory_allocated() / 1024**3
            ),
            "session_elapsed_minutes": (
                time.perf_counter() - training_start_time
            ) / 60,
        }
        histories["steps"].append(step_record)

        if step % LOG_INTERVAL == 0:
            recent_records = histories["steps"][-LOG_INTERVAL:]
            recent_wall = [
                record["wall_step_ms"]
                for record in recent_records
            ]
            recent_gpu = [
                record["gpu_step_ms"]
                for record in recent_records
            ]
            recent_throughput = [
                record["wall_tokens_per_second"]
                for record in recent_records
            ]

            system_record = {
                "step": step,
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                **query_nvidia_smi(),
            }
            histories["system"].append(system_record)

            print(
                f"step {step:>7,} | "
                f"loss {mean_training_loss:7.4f} | "
                f"lr {learning_rate:.2e} | "
                f"median {statistics.median(recent_wall):7.1f} ms | "
                f"P95 {percentile(recent_wall, 0.95):7.1f} ms | "
                f"{statistics.fmean(recent_throughput):,.0f} tok/s | "
                f"GPU median {statistics.median(recent_gpu):7.1f} ms | "
                f"VRAM {step_record['allocated_vram_gb']:.2f}/"
                f"{step_record['reserved_vram_gb']:.2f} GB"
            )

            if "gpu_utilization_percent" in system_record:
                print(
                    f"  NVIDIA | util "
                    f"{system_record['gpu_utilization_percent']:.0f}% | "
                    f"{system_record['temperature_c']:.0f} C | "
                    f"{system_record['power_draw_w']:.1f} W / "
                    f"{system_record['power_limit_w']:.1f} W"
                )

        if step % EVAL_INTERVAL == 0:
            losses = estimate_loss()

            eval_record = {
                "step": step,
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "train_loss": losses["train"],
                "val_loss": losses["val"],
                "wall_seconds": losses["wall_seconds"],
                "gpu_seconds": losses["gpu_seconds"],
                "tokens_evaluated": losses["tokens_evaluated"],
                "wall_tokens_per_second": (
                    losses["wall_tokens_per_second"]
                ),
                "gpu_tokens_per_second": (
                    losses["gpu_tokens_per_second"]
                ),
                "learning_rate": learning_rate,
                "session_elapsed_minutes": (
                    time.perf_counter() - training_start_time
                ) / 60,
                "peak_vram_gb": (
                    torch.cuda.max_memory_allocated() / 1024**3
                ),
            }
            histories["evaluations"].append(eval_record)

            print(
                f"  EVAL | train {losses['train']:.4f} | "
                f"validation {losses['val']:.4f} | "
                f"{losses['wall_seconds']:.2f}s | "
                f"{losses['wall_tokens_per_second']:,.0f} tok/s"
            )

        if step > start_step and step % GENERATE_INTERVAL == 0:
            model.eval()
            seed = torch.tensor(
                [[stoi.get("\n", 0)]],
                dtype=torch.long,
                device=device,
            )

            torch.cuda.synchronize()
            generation_wall_start = time.perf_counter()
            generation_gpu_start = torch.cuda.Event(enable_timing=True)
            generation_gpu_end = torch.cuda.Event(enable_timing=True)
            generation_gpu_start.record()

            generated = model.generate(
                seed,
                max_new_tokens=GENERATION_SAMPLE_TOKENS,
                temperature=0.8,
                top_k=40,
            )

            generation_gpu_end.record()
            torch.cuda.synchronize()

            generation_wall_seconds = (
                time.perf_counter() - generation_wall_start
            )
            generation_gpu_seconds = (
                generation_gpu_start.elapsed_time(
                    generation_gpu_end
                ) / 1000
            )

            generation_record = {
                "step": step,
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "generated_tokens": GENERATION_SAMPLE_TOKENS,
                "wall_seconds": generation_wall_seconds,
                "gpu_seconds": generation_gpu_seconds,
                "wall_tokens_per_second": (
                    GENERATION_SAMPLE_TOKENS
                    / max(generation_wall_seconds, 1e-9)
                ),
                "gpu_tokens_per_second": (
                    GENERATION_SAMPLE_TOKENS
                    / max(generation_gpu_seconds, 1e-9)
                ),
            }
            histories["generations"].append(generation_record)

            print("\n--- Generated sample ---")
            print(decode(generated[0].tolist()))
            print(
                f"--- {generation_record['wall_tokens_per_second']:.1f} "
                "generated tokens/s ---\n"
            )

            model.train()
            del seed, generated

        if step > start_step and step % CHECKPOINT_INTERVAL == 0:
            save_checkpoint(
                step=step,
                histories=histories,
                reason="scheduled",
            )

        step += 1

except KeyboardInterrupt:
    print("\nTraining interrupted by user.")
    save_checkpoint(
        step=max(step - 1, 0),
        histories=histories,
        reason="interrupted",
    )

except torch.cuda.OutOfMemoryError:
    print("\nCUDA ran out of memory.")
    print(
        "Reduce config['micro_batch_size'] or select a smaller profile, "
        "restart the kernel, and rerun."
    )
    save_checkpoint(
        step=max(step - 1, 0),
        histories=histories,
        reason="out_of_memory",
    )
    raise

else:
    save_checkpoint(
        step=max(step - 1, 0),
        histories=histories,
        reason="completed",
    )

finally:
    total_session_seconds = (
        time.perf_counter() - training_start_time
    )
    histories_to_csv(histories)

    print(f"Session training time: {total_session_seconds / 60:.2f} minutes")
    print(f"Session tokens processed: {session_tokens_processed:,}")

    if histories["steps"]:
        session_step_records = [
            record
            for record in histories["steps"]
            if record["step"] >= start_step
        ]
        wall_values = [
            record["wall_step_ms"]
            for record in session_step_records
        ]
        throughput_values = [
            record["wall_tokens_per_second"]
            for record in session_step_records
        ]

        print("Median optimizer-step time:", round(statistics.median(wall_values), 2), "ms")
        print("P95 optimizer-step time:", round(percentile(wall_values, 0.95), 2), "ms")
        print(
            "Mean training throughput:",
            f"{statistics.fmean(throughput_values):,.0f}",
            "tokens/s",
        )

## 11. Benchmark tables and charts

In [ ]:
steps_df = pd.DataFrame(histories.get("steps", []))
evaluations_df = pd.DataFrame(histories.get("evaluations", []))
generations_df = pd.DataFrame(histories.get("generations", []))
system_df = pd.DataFrame(histories.get("system", []))
checkpoints_df = pd.DataFrame(histories.get("checkpoints", []))
microbenchmarks_df = pd.DataFrame(
    histories.get("microbenchmarks", [])
)

print("Latest optimizer-step records:")
if not steps_df.empty:
    display(steps_df.tail(10))
else:
    print("No optimizer-step metrics recorded yet.")

print("\nLatest evaluation records:")
if not evaluations_df.empty:
    display(evaluations_df.tail(10))
else:
    print("No evaluation metrics recorded yet.")

print("\nModel microbenchmarks:")
if not microbenchmarks_df.empty:
    display(microbenchmarks_df)
else:
    print("No microbenchmark metrics recorded yet.")

if not steps_df.empty:
    plt.figure(figsize=(10, 6))
    plt.plot(steps_df["step"], steps_df["wall_step_ms"])
    plt.xlabel("Training step")
    plt.ylabel("Wall-clock step time (ms)")
    plt.title("Optimizer-step time")
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(10, 6))
    plt.plot(
        steps_df["step"],
        steps_df["wall_tokens_per_second"],
    )
    plt.xlabel("Training step")
    plt.ylabel("Tokens per second")
    plt.title("Training throughput")
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(10, 6))
    plt.plot(
        steps_df["step"],
        steps_df["peak_vram_gb"],
    )
    plt.xlabel("Training step")
    plt.ylabel("Peak allocated VRAM (GB)")
    plt.title("VRAM usage")
    plt.grid(True, alpha=0.3)
    plt.show()

if not evaluations_df.empty:
    plt.figure(figsize=(10, 6))
    plt.plot(
        evaluations_df["step"],
        evaluations_df["train_loss"],
        label="Training loss",
    )
    plt.plot(
        evaluations_df["step"],
        evaluations_df["val_loss"],
        label="Validation loss",
    )
    plt.xlabel("Training step")
    plt.ylabel("Cross-entropy loss")
    plt.title("GPT learning curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

if (
    not system_df.empty
    and "temperature_c" in system_df.columns
):
    plt.figure(figsize=(10, 6))
    plt.plot(
        system_df["step"],
        system_df["temperature_c"],
    )
    plt.xlabel("Training step")
    plt.ylabel("GPU temperature (C)")
    plt.title("GPU temperature during training")
    plt.grid(True, alpha=0.3)
    plt.show()

if (
    not system_df.empty
    and "power_draw_w" in system_df.columns
):
    plt.figure(figsize=(10, 6))
    plt.plot(
        system_df["step"],
        system_df["power_draw_w"],
    )
    plt.xlabel("Training step")
    plt.ylabel("GPU power draw (W)")
    plt.title("GPU power draw during training")
    plt.grid(True, alpha=0.3)
    plt.show()

## 12. Generate text and benchmark inference

In [ ]:
prompt = "ROMEO:\n"
max_new_tokens = 1_000
temperature = 0.8
top_k = 40

unknown_characters = [
    character
    for character in prompt
    if character not in stoi
]

if unknown_characters:
    raise ValueError(
        f"Prompt contains characters outside the training vocabulary: "
        f"{sorted(set(unknown_characters))}"
    )

prompt_tokens = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device,
)

model.eval()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

generation_wall_start = time.perf_counter()
generation_gpu_start = torch.cuda.Event(enable_timing=True)
generation_gpu_end = torch.cuda.Event(enable_timing=True)
generation_gpu_start.record()

generated_tokens = model.generate(
    prompt_tokens,
    max_new_tokens=max_new_tokens,
    temperature=temperature,
    top_k=top_k,
)

generation_gpu_end.record()
torch.cuda.synchronize()

generation_wall_seconds = (
    time.perf_counter() - generation_wall_start
)
generation_gpu_seconds = (
    generation_gpu_start.elapsed_time(generation_gpu_end) / 1000
)

generated_text = decode(generated_tokens[0].tolist())
print(generated_text)

print("\nInference benchmark")
print("Generated tokens:", f"{max_new_tokens:,}")
print("Wall time:", round(generation_wall_seconds, 3), "seconds")
print("GPU-event time:", round(generation_gpu_seconds, 3), "seconds")
print(
    "Wall throughput:",
    round(max_new_tokens / generation_wall_seconds, 2),
    "tokens/s",
)
print(
    "GPU throughput:",
    round(max_new_tokens / generation_gpu_seconds, 2),
    "tokens/s",
)
print(
    "Peak VRAM:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB",
)

model.train()

## 13. Final benchmark summary

In [ ]:
steps_df = pd.DataFrame(histories.get("steps", []))
evaluations_df = pd.DataFrame(histories.get("evaluations", []))
generations_df = pd.DataFrame(histories.get("generations", []))
system_df = pd.DataFrame(histories.get("system", []))
checkpoints_df = pd.DataFrame(histories.get("checkpoints", []))

summary = {
    "gpu": gpu_name,
    "vram_gb": round(total_vram_gb, 2),
    "pytorch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "mixed_precision": str(amp_dtype),
    "profile": selected_profile,
    "parameters": parameter_count,
    "layers": model_config.n_layer,
    "heads": model_config.n_head,
    "embedding_dimension": model_config.n_embd,
    "context_length": model_config.block_size,
    "micro_batch_size": micro_batch_size,
    "gradient_accumulation_steps": gradient_accumulation_steps,
    "effective_tokens_per_step": effective_tokens_per_step,
    "recorded_optimizer_steps": len(steps_df),
    "total_recorded_training_tokens": (
        int(steps_df["effective_tokens"].sum())
        if not steps_df.empty
        else 0
    ),
    "peak_vram_gb": (
        float(steps_df["peak_vram_gb"].max())
        if not steps_df.empty
        else torch.cuda.max_memory_allocated() / 1024**3
    ),
    "checkpoint": str(CHECKPOINT_PATH.resolve()),
}

if not steps_df.empty:
    wall_stats = distribution_stats(
        steps_df["wall_step_ms"].tolist(),
        "optimizer_step_ms",
    )
    throughput_stats = distribution_stats(
        steps_df["wall_tokens_per_second"].tolist(),
        "training_tokens_per_second",
    )
    tflops_stats = distribution_stats(
        steps_df["approximate_gpu_tflops"].tolist(),
        "approximate_gpu_tflops",
    )
    summary.update(wall_stats)
    summary.update(throughput_stats)
    summary.update(tflops_stats)

if not evaluations_df.empty:
    summary.update({
        "initial_train_loss": float(
            evaluations_df.iloc[0]["train_loss"]
        ),
        "final_train_loss": float(
            evaluations_df.iloc[-1]["train_loss"]
        ),
        "initial_validation_loss": float(
            evaluations_df.iloc[0]["val_loss"]
        ),
        "final_validation_loss": float(
            evaluations_df.iloc[-1]["val_loss"]
        ),
        "mean_evaluation_seconds": float(
            evaluations_df["wall_seconds"].mean()
        ),
    })

if (
    not generations_df.empty
    and "wall_tokens_per_second" in generations_df.columns
):
    summary["mean_generation_tokens_per_second"] = float(
        generations_df["wall_tokens_per_second"].mean()
    )

if (
    not system_df.empty
    and "gpu_utilization_percent" in system_df.columns
):
    summary.update({
        "mean_gpu_utilization_percent": float(
            system_df["gpu_utilization_percent"].mean()
        ),
        "max_temperature_c": float(
            system_df["temperature_c"].max()
        ),
        "mean_power_draw_w": float(
            system_df["power_draw_w"].mean()
        ),
        "max_power_draw_w": float(
            system_df["power_draw_w"].max()
        ),
    })

if not checkpoints_df.empty:
    summary.update({
        "mean_checkpoint_seconds": float(
            checkpoints_df["checkpoint_seconds"].mean()
        ),
        "latest_checkpoint_size_mb": float(
            checkpoints_df.iloc[-1]["checkpoint_size_mb"]
        ),
    })

summary_df = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in summary.items()]
)
summary_df.to_csv(SUMMARY_PATH, index=False)

display(summary_df)
print("Summary saved to:", SUMMARY_PATH.resolve())

## Reading the benchmark correctly

For repeatable comparisons between computers or settings:

1. Keep the same profile, context length, micro-batch size and accumulation count.
2. Compare runs after the warm-up period rather than the first few steps.
3. Use median and P95 step time, not only the single fastest result.
4. Compare both wall-clock throughput and GPU-event throughput.
5. Note the power limit because laptop and workstation Blackwell cards can run
   at very different wattages.
6. Record driver, PyTorch and CUDA-runtime versions.
7. Use validation loss to confirm the machine is doing meaningful learning,
   rather than only producing high synthetic throughput.
8. Check whether VRAM is nearly full. A lightly loaded GPU is not a fair measure
   of its maximum capability.

The `gpt_output` folder contains the checkpoint plus CSV files for optimizer
steps, validation, generation, system telemetry, checkpoint timing,
microbenchmarks and the final summary.